In [9]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Create a header to mimic a real browser
header = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# SOURCE 1: Economy of Oklahoma
url_econ = "https://en.wikipedia.org/wiki/Economy_of_Oklahoma"
resp_econ = requests.get(url_econ, headers=header)
tables = pd.read_html(resp_econ.text)

# Grabbing the Top Employers table (usually index 3)
df = tables[3].copy()
df.columns = ['Rank', 'Employer', 'Employees']

print("--- STEP 1: RAW DATA FROM WIKIPEDIA ---")
print(df.head(10))

--- STEP 1: RAW DATA FROM WIKIPEDIA ---
   Rank                               Employer  Employees
0     1    United States Department of Defense      69000
1     2                                Walmart      32200
2     3                 University of Oklahoma      17800
3     4                       Chickasaw Nation      11300
4     5             Choctaw Nation of Oklahoma      10000
5     6                        Integris Health       8900
6     7                        Cherokee Nation       8500
7     8              Oklahoma State University       8200
8     9           United States Postal Service       6900
9    10  Oklahoma Department of Human Services       6600


C:\Users\bibek\AppData\Local\Temp\ipykernel_32892\3236875608.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp_econ.text)


In [37]:
# 1. Force the column to be a string (text) so we can use .str.replace
df['Employees'] = df['Employees'].astype(str)

# 2. Regex will work now
# Remove Wikipedia citations like [1] or [note 1]
df['Employees_Clean'] = df['Employees'].str.replace(r'\[.*\]', '', regex=True)

# 3. Remove commas 
df['Employees_Clean'] = df['Employees_Clean'].str.replace(',', '')

# 4. Convert it back to a real number so we can do math
df['Employees_Clean'] = pd.to_numeric(df['Employees_Clean'], errors='coerce')

print(" CLEANED NUMERIC DATA ")
print(df[['Employer', 'Employees_Clean']].head(10))

 CLEANED NUMERIC DATA 
                                Employer  Employees_Clean
0    United States Department of Defense            69000
1                                Walmart            32200
2                 University of Oklahoma            17800
3                       Chickasaw Nation            11300
4             Choctaw Nation of Oklahoma            10000
5                        Integris Health             8900
6                        Cherokee Nation             8500
7              Oklahoma State University             8200
8           United States Postal Service             6900
9  Oklahoma Department of Human Services             6600


In [15]:
# SOURCE 2: Companies based in OKC
url_hqs = "https://en.wikipedia.org/wiki/List_of_companies_based_in_Oklahoma_City"
resp_hqs = requests.get(url_hqs, headers=header)
soup = BeautifulSoup(resp_hqs.text, 'html.parser')

# Extract every bullet point (li) on the page to find company names
hq_list = [li.text.strip() for li in soup.find_all('li')]

print("--- STEP 3: DATA FROM SECOND SOURCE (OKC HQ LIST) ---")
print(hq_list[25:35]) # Showing a slice of the scraped list

--- STEP 3: DATA FROM SECOND SOURCE (OKC HQ LIST) ---
['Talk', 'Read', 'Edit', 'View history', 'Read', 'Edit', 'View history', 'What links here', 'Related changes', 'Upload file']


In [39]:
import pandas as pd

# 1. THE COMPLETE TRUTH LIST
okc_hq_registry = [
    "Hobby Lobby", 
    "Integris Health", 
    "Defense",           
    "Oklahoma",          
    "Paycom", 
    "Devon Energy", 
    "OGE Energy", 
    "Braum's", 
    "Love's Travel Stops", 
    "Continental Resources", 
    "American Fidelity", 
    "BancFirst", 
    "Sonic Drive-In"
]

# 2. THE LOGIC FUNCTION
def verify_okc_local(employer_name):
    employer_str = str(employer_name)
    # This checks if any word in our truth list appears in the Employer's name
    if any(hq.lower() in employer_str.lower() for hq in okc_hq_registry):
        return "Yes"
    else:
        return "No"

# 3. APPLYing TO THE DATASET
# This creates the final column based on my clean truth data
df_final['Headquartered_in_OKC'] = df_final['Employer'].apply(verify_okc_local)
print("--- FINAL DATASET PREVIEW ---")
print(df_final.head(20))

--- FINAL DATASET PREVIEW ---
    Rank                                      Employer  Total_Employees  \
0      1           United States Department of Defense            69000   
1      2                                       Walmart            32200   
2      3                        University of Oklahoma            17800   
3      4                              Chickasaw Nation            11300   
4      5                    Choctaw Nation of Oklahoma            10000   
5      6                               Integris Health             8900   
6      7                               Cherokee Nation             8500   
7      8                     Oklahoma State University             8200   
8      9                  United States Postal Service             6900   
9     10         Oklahoma Department of Human Services             6600   
10    11                                   Hobby Lobby             6401   
11    12                                  Mercy Health             630

In [35]:
# Saving the final version 
filename = "OKC_Employment_Connectivity_Index_FINAL.csv"

# index=False ensures we don't get an extra 'unnamed' column in Excel
df_final.to_csv(filename, index=False, encoding='utf-8-sig')

print(f" Success! Your file '{filename}' has been created.")

 Success! Your file 'OKC_Employment_Connectivity_Index_FINAL.csv' has been created.
